In [ ]:
# -*- coding: utf-8 -*-

import zipfile
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/high_plains_quifer.zip'
extract_path = '/content/ogallala_shp'

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted to:", extract_path)

# !pip install -q xee xarray netcdf4 geopandas pyproj dask h5netcdf

# # ============================================================================
# # SMAP L3 (SPL3SMP_E) Extractor — Batch Validation & Redownload Logic
# # ============================================================================
!pip -q install earthaccess geopandas shapely h5py netCDF4 2>&1 | tail -2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files extracted to: /content/ogallala_shp


In [ ]:
import os
import re
import gc
import glob
import time
import shutil
import logging
import hashlib
from datetime import datetime, timezone, timedelta

import numpy as np
import h5py
import netCDF4 as nc4

try:
    import geopandas as gpd
except ImportError as e:
    raise ImportError("geopandas is required: pip install geopandas") from e
try:
    from shapely.vectorized import contains as vec_contains
except ImportError:
    vec_contains = None  # fallback to prepared-geometry loop below
import earthaccess

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("smap_l3e")

# ============================================================================
# CONFIGURATION
# ============================================================================
SHORT_NAME = "SPL3SMP_E"

# -- Edit these for your environment ---------------------------------------
# Colab defaults:
SHAP_PATH = "/content/ogallala_shp/high_plains_quifer/hp_bound2010.shp"
DRIVE_OUT_DIR = "/content/drive/MyDrive/SMAP_L3E_Ogallala"
LOCAL_WORK = "/content/smap_raw"
LOCAL_NC = "/content/smap_nc"
# Local Windows alternative (uncomment to use):
# SHAP_PATH     = r"G:\MSU_GWB\datasets\high_plains_quifer\hp_bound2010.shp"
# DRIVE_OUT_DIR = r"G:\MSU_GWB\datasets\SMAP_L3E_Ogallala"
# LOCAL_WORK    = r"G:\MSU_GWB\datasets\smap_raw_l3e"
# LOCAL_NC      = r"G:\MSU_GWB\datasets\smap_nc_l3e"

SINGLE_FILE_MODE = True
OUTPUT_FILENAME = "SPL3SMP_E_Ogallala_FULL.nc"

MISSION_START = datetime(2015, 3, 31, 12, 0)  # first SPL3SMP_E day (noon UTC)
TIME_UNITS = "days since 2015-03-31 12:00:00 UTC"
TIME_CALENDAR = "standard"  # CF-compliant; 'gregorian'/'standard' equivalent

FILL_FLOAT = np.float32(-9999.0)
FILL_UINT16 = np.uint16(65534)
FILL_UINT8 = np.uint8(254)

BATCH_DATES = 150     # daily files per search batch
FLUSH_DATES = 25      # sync to disk every N days
BACKUP_DATES = 75     # copy monolithic file to Drive every N days
CHECKPOINT_SECONDS = 3600  # wall-clock interval for Drive runtime checkpoints (1 hour)
MAX_RETRIES = 4
N_THREADS = 12
COMPRESS_LEVEL = 3

# Matches: SMAP_L3_SM_P_E_20150403_R17400_001.h5
SMAP_RE = re.compile(r"SMAP_L3_SM_P_E_(\d{8})_R.*\.h5$")
RUNTIME_BKUP_RE = re.compile(r"SMAP_L3E_ogallala_(\d+)hr\.nc$")

AM_GROUP = "Soil_Moisture_Retrieval_Data_AM"
PM_GROUP = "Soil_Moisture_Retrieval_Data_PM"

# (output_name, hdf_group, [candidate dataset names in preference order])
# NOTE: concrete *_dca names come FIRST because the bare pointer names
# (soil_moisture, vegetation_opacity, retrieval_qual_flag, ...) can be
# HDF5 object references rather than plain arrays.
FLOAT_TARGETS = [
    ("soil_moisture_am", AM_GROUP, ["soil_moisture_dca", "soil_moisture"]),
    ("soil_moisture_pm", PM_GROUP, ["soil_moisture_dca_pm", "soil_moisture_pm",
                                    "soil_moisture_dca", "soil_moisture"]),
    ("soil_moisture_error_am", AM_GROUP, ["soil_moisture_error", "soil_moisture_error_dca"]),
    ("soil_moisture_error_pm", PM_GROUP, ["soil_moisture_error_pm",
                                          "soil_moisture_error_dca_pm",
                                          "soil_moisture_error", "soil_moisture_error_dca"]),
    # VOD (vegetation optical depth / tau, DCA joint retrieval with SM)
    ("vod_am", AM_GROUP, ["vegetation_opacity_dca", "vegetation_opacity"]),
    ("vod_pm", PM_GROUP, ["vegetation_opacity_dca_pm", "vegetation_opacity_pm",
                          "vegetation_opacity_dca", "vegetation_opacity"]),
    ("vod_error_am", AM_GROUP, ["vegetation_opacity_error_dca", "vegetation_opacity_error",
                                "vegetation_opacity_dca_error", "vegetation_opacity_error_dca"]),
    ("vod_error_pm", PM_GROUP, ["vegetation_opacity_error_dca_pm", "vegetation_opacity_error_pm",
                                "vegetation_opacity_error", "vegetation_opacity_dca"]),
]

FLAG_TARGETS = [
    ("retrieval_qual_flag_am", AM_GROUP, ["retrieval_qual_flag_dca", "retrieval_qual_flag"]),
    ("retrieval_qual_flag_pm", PM_GROUP, ["retrieval_qual_flag_dca_pm", "retrieval_qual_flag_pm",
                                          "retrieval_qual_flag_dca", "retrieval_qual_flag"]),
    ("surface_flag_am", AM_GROUP, ["surface_flag", "surface_qual_flag"]),
    ("surface_flag_pm", PM_GROUP, ["surface_flag_pm", "surface_flag",
                                   "surface_qual_flag_pm", "surface_qual_flag"]),
]

# CF metadata for output variables. 'ancillary' links SM/VOD -> QA flags.
VAR_META = {
    "soil_moisture_am": dict(long_name="Volume fraction of water in surface soil (AM descending, DCA)",
                             standard_name="volume_fraction_of_water_in_soil",
                             units="m3 m-3", valid_min=np.float32(0.02),
                             valid_max=np.float32(0.7),
                             ancillary="retrieval_qual_flag_am surface_flag_am"),
    "soil_moisture_pm": dict(long_name="Volume fraction of water in surface soil (PM ascending, DCA)",
                             standard_name="volume_fraction_of_water_in_soil",
                             units="m3 m-3", valid_min=np.float32(0.02),
                             valid_max=np.float32(0.7),
                             ancillary="retrieval_qual_flag_pm surface_flag_pm"),
    "soil_moisture_error_am": dict(long_name="Uncertainty of surface soil moisture retrieval (AM)",
                                   units="m3 m-3", valid_min=np.float32(0.0),
                                   valid_max=np.float32(0.5),
                                   ancillary="retrieval_qual_flag_am surface_flag_am"),
    "soil_moisture_error_pm": dict(long_name="Uncertainty of surface soil moisture retrieval (PM)",
                                   units="m3 m-3", valid_min=np.float32(0.0),
                                   valid_max=np.float32(0.5),
                                   ancillary="retrieval_qual_flag_pm surface_flag_pm"),
    # VOD has no CF standard_name; long_name only (CF forbids inventing one).
    "vod_am": dict(long_name="Vegetation optical depth at L-band (AM descending, DCA joint retrieval)",
                   units="1", valid_min=np.float32(0.0), valid_max=np.float32(5.0),
                   ancillary="retrieval_qual_flag_am surface_flag_am"),
    "vod_pm": dict(long_name="Vegetation optical depth at L-band (PM ascending, DCA joint retrieval)",
                   units="1", valid_min=np.float32(0.0), valid_max=np.float32(5.0),
                   ancillary="retrieval_qual_flag_pm surface_flag_pm"),
    "vod_error_am": dict(long_name="Uncertainty of vegetation optical depth (AM)",
                         units="1", valid_min=np.float32(0.0), valid_max=np.float32(5.0),
                         ancillary="retrieval_qual_flag_am surface_flag_am"),
    "vod_error_pm": dict(long_name="Uncertainty of vegetation optical depth (PM)",
                         units="1", valid_min=np.float32(0.0), valid_max=np.float32(5.0),
                         ancillary="retrieval_qual_flag_pm surface_flag_pm"),
}

FLAG_META = {
    "retrieval_qual_flag_am": dict(
        long_name="SMAP L3E retrieval quality flag (AM). Bit 0 = recommended quality (0=good).",
        meanings="not_recommended_quality retrieval_attempted frozen_ground snow_ice "
                 "slope_correction precipitation_correction urban static_water radar_water",
        masks="1 2 4 8 16 32 64 128 256 512"),
    "retrieval_qual_flag_pm": dict(
        long_name="SMAP L3E retrieval quality flag (PM). Bit 0 = recommended quality (0=good).",
        meanings="not_recommended_quality retrieval_attempted frozen_ground snow_ice "
                 "slope_correction precipitation_correction urban static_water radar_water",
        masks="1 2 4 8 16 32 64 128 256 512"),
    "surface_flag_am": dict(
        long_name="SMAP L3E surface condition flag (AM); encodes open water, precip, snow, frozen ground, slope, urban.",
        meanings="open_water urban precipitation snow frozen_ground slope",
        masks="3 4 16 32 192 1024"),
    "surface_flag_pm": dict(
        long_name="SMAP L3E surface condition flag (PM); encodes open water, precip, snow, frozen ground, slope, urban.",
        meanings="open_water urban precipitation snow frozen_ground slope",
        masks="3 4 16 32 192 1024"),
}

EASE2_WKT = (
    'PROJCRS["WGS 84 / NSIDC EASE-Grid 2.0 Global",'
    'BASEGEOGCRS["WGS 84",DATUM["World Geodetic System 1984",'
    'ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]]],'
    'PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]]],'
    'CONVERSION["NSIDC EASE-Grid 2.0 Global",'
    'METHOD["Lambert Cylindrical Equal Area (Spherical)",ID["EPSG",9834]],'
    'PARAMETER["Latitude of 1st standard parallel",30,ANGLEUNIT["degree",0.0174532925199433]],'
    'PARAMETER["Longitude of natural origin",0,ANGLEUNIT["degree",0.0174532925199433]],'
    'PARAMETER["False easting",0,LENGTHUNIT["metre",1]],'
    'PARAMETER["False northing",0,LENGTHUNIT["metre",1]]],'
    'CS[Cartesian,2],AXIS["easting (X)",east],AXIS["northing (Y)",north],'
    'LENGTHUNIT["metre",1],ID["EPSG",6933]]'
)
EASE2_PROJ4 = "+proj=cea +lon_0=0 +lat_ts=30 +x_0=0 +y_0=0 +ellps=WGS84 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs"

for _d in (LOCAL_WORK, LOCAL_NC, DRIVE_OUT_DIR):
    os.makedirs(_d, exist_ok=True)


# ============================================================================
# UTILITIES
# ============================================================================
def parse_dt(filename):
    """Parse daily date from SPL3SMP_E filename -> datetime at 12:00 UTC."""
    m = SMAP_RE.search(os.path.basename(filename))
    if not m:
        return None
    return datetime.strptime(m.group(1), "%Y%m%d").replace(
        hour=12, minute=0, second=0, tzinfo=None)


def get_latest_drive_backup(drive_dir):
    """Most recent runtime checkpoint or FULL file in Drive (by mtime).

    Case-insensitive: matches SPL3SMP_E_Ogallala_FULL.nc,
    SPL3SMP_E_Ogallala_Full.nc, SMAP_L3E_ogallala_*hr.nc, etc.
    This matters on Colab (Linux, case-sensitive) after a restart where
    only the Drive copy survives.
    """
    if not os.path.isdir(drive_dir):
        log.warning("Drive output dir not found: %s (did you mount Drive?)", drive_dir)
        return None
    try:
        listing = os.listdir(drive_dir)
    except Exception as e:
        log.warning("Cannot list Drive dir %s: %s", drive_dir, e)
        return None
    log.info("Drive dir %s contains %d files; NC files: %s",
             drive_dir, len(listing),
             sorted([f for f in listing if f.lower().endswith(".nc")])[:20])

    hr_cands = []
    full_cands = []
    for fname in listing:
        fl = fname.lower()
        fpath = os.path.join(drive_dir, fname)
        if RUNTIME_BKUP_RE.search(fname):
            try:
                hr_cands.append((os.path.getmtime(fpath), fpath))
            except OSError:
                continue
        # Any full-file variant: contains both 'ogallala' and '.nc', not a hr checkpoint
        if fl.endswith(".nc") and "ogallala" in fl and "hr.nc" not in fl:
            # Prefer names containing spl3smp_e / smap, but accept any ogallala nc
            # so Full vs FULL vs other capitalizations all match.
            try:
                full_cands.append((os.path.getmtime(fpath), fpath))
            except OSError:
                continue
    # Exact configured name first (fast path), then fuzzy matches
    exact = os.path.join(drive_dir, OUTPUT_FILENAME)
    cands = []
    if os.path.exists(exact):
        try:
            cands.append((os.path.getmtime(exact), exact))
        except OSError:
            pass
    cands.extend(hr_cands)
    cands.extend(full_cands)
    if not cands:
        log.warning("No backup *.nc found in Drive dir %s.", drive_dir)
        return None
    # Deduplicate (exact may also appear in full_cands), keep newest
    seen, uniq = set(), []
    for mt, p in sorted(cands, reverse=True):
        if p not in seen:
            seen.add(p)
            uniq.append((mt, p))
    best = uniq[0][1]
    log.info("Latest Drive backup selected: %s (mtime %s)",
             os.path.basename(best),
             datetime.fromtimestamp(os.path.getmtime(best)).isoformat())
    return best


def read_last_timestep(nc_path):
    """Return (last_datetime_noon, n_times) from an output file, or (None, 0)."""
    try:
        with nc4.Dataset(nc_path, "r") as ds:
            if "time" not in ds.variables:
                return None, 0
            n_times = len(ds.variables["time"])
            if n_times == 0:
                return None, 0
            last_val = float(ds.variables["time"][-1])
            last_dt = nc4.num2date(last_val, ds.variables["time"].units,
                                   ds.variables["time"].calendar)
            last_dt = datetime(last_dt.year, last_dt.month, last_dt.day, 12, 0)
            return last_dt, n_times
    except Exception as e:
        log.warning("Could not read last timestep from %s: %s", nc_path, e)
        return None, 0


def safe_search(start_dt, end_dt, retries=3):
    """earthaccess.search_data with retries; returns list (possibly empty)."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            results = earthaccess.search_data(
                short_name=SHORT_NAME,
                temporal=(start_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
                          end_dt.strftime("%Y-%m-%dT%H:%M:%SZ")))
            return list(results) if results else []
        except Exception as e:
            last_err = e
            log.warning("Search attempt %d/%d for %s..%s failed: %s",
                        attempt, retries, start_dt.date(), end_dt.date(), e)
            time.sleep(10 * attempt)
    log.error("Search failed after %d attempts: %s", retries, last_err)
    return []


def safe_download(results, dest_dir, threads=1):
    """earthaccess.download that never raises on empty input; returns list of paths."""
    if not results:
        return []
    try:
        out = earthaccess.download(results, local_path=dest_dir, threads=threads)
        return [str(f) for f in out] if out else []
    except ValueError as e:
        # earthaccess raises "List of URLs or DataGranule instances expected"
        # when given an empty/None list — treat as no downloads.
        log.warning("Download skipped (empty granule list): %s", e)
        return []
    except Exception as e:
        log.error("Download failed: %s", e)
        return []


def load_roi(shp_path, deg=0.1):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    roi = gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union
    return roi.simplify(deg, preserve_topology=True)


def _list_h5_datasets(h5obj, prefix=""):
    """Recursively list dataset paths (for debug logging)."""
    out = []
    for k in h5obj.keys():
        p = f"{prefix}/{k}" if prefix else f"/{k}"
        try:
            obj = h5obj[k]
        except Exception:
            continue
        if isinstance(obj, h5py.Dataset):
            out.append((p, obj.shape, str(obj.dtype)))
        elif isinstance(obj, h5py.Group):
            out.extend(_list_h5_datasets(obj, p))
    return out


def _find_latlon(h5_path):
    """Locate 2D latitude/longitude datasets in an SPL3SMP_E granule.

    Tries per-group 'latitude'/'longitude' first (the documented layout),
    then common fallbacks. Returns (lat_path, lon_path) with leading '/'.
    """
    candidates = [
        (f"/{AM_GROUP}/latitude", f"/{AM_GROUP}/longitude"),
        (f"/{PM_GROUP}/latitude_pm", f"/{PM_GROUP}/longitude_pm"),
        (f"/{PM_GROUP}/latitude", f"/{PM_GROUP}/longitude"),
        ("/cell_lat", "/cell_lon"),
        ("/latitude", "/longitude"),
    ]
    with h5py.File(h5_path, "r") as f:
        for lat_p, lon_p in candidates:
            if lat_p in f and lon_p in f:
                ds_lat, ds_lon = f[lat_p], f[lon_p]
                if ds_lat.ndim == 2 and ds_lon.ndim == 2 and ds_lat.shape == ds_lon.shape:
                    return lat_p, lon_p, ds_lat.shape, str(ds_lat.dtype)
        # Nothing matched: log layout to help the user diagnose version drift.
        log.warning("Lat/lon auto-detect failed; file layout:")
        for p, sh, dt in _list_h5_datasets(f)[:60]:
            log.warning("  %s shape=%s dtype=%s", p, sh, dt)
    raise KeyError(
        "Could not locate 2D latitude/longitude in SPL3SMP_E file. "
        f"Tried {[c[0] for c in candidates]}. See layout log above.")


def build_mask(sample_h5, roi):
    lat_path, lon_path, shape, dtype = _find_latlon(sample_h5)
    log.info("Using geolocation: %s / %s shape=%s dtype=%s", lat_path, lon_path, shape, dtype)
    with h5py.File(sample_h5, "r") as f:
        cell_lat = f[lat_path][:]
        cell_lon = f[lon_path][:]

    # SPL3SMP_E grid fill is -9999.0; exclude from bbox math safely.
    valid_geo = np.isfinite(cell_lat) & np.isfinite(cell_lon) & (cell_lat > -900) & (cell_lon > -9000)
    minx, miny, maxx, maxy = roi.bounds
    bbox_mask = (valid_geo & (cell_lat >= miny) & (cell_lat <= maxy)
                 & (cell_lon >= minx) & (cell_lon <= maxx))
    rows = np.where(np.any(bbox_mask, axis=1))[0]
    cols = np.where(np.any(bbox_mask, axis=0))[0]
    if len(rows) == 0 or len(cols) == 0:
        raise ValueError("No EASE-2 grid cells fall within the ROI bounding box.")
    r0, r1 = int(rows[0]), int(rows[-1]) + 1
    c0, c1 = int(cols[0]), int(cols[-1]) + 1

    lat_sub = cell_lat[r0:r1, c0:c1]
    lon_sub = cell_lon[r0:r1, c0:c1]

    if vec_contains is not None:
        mask_2d = vec_contains(roi, lon_sub, lat_sub)
    else:  # shapely>=2 removed shapely.vectorized; use prepared geometry loop
        from shapely.prepared import prep
        from shapely.geometry import Point
        roi_prep = prep(roi)
        ny, nx = lat_sub.shape
        mask_2d = np.zeros((ny, nx), dtype=bool)
        sub = bbox_mask[r0:r1, c0:c1]
        for iy in range(ny):
            for ix in range(nx):
                if sub[iy, ix] and np.isfinite(lat_sub[iy, ix]):
                    if roi_prep.contains(Point(float(lon_sub[iy, ix]), float(lat_sub[iy, ix]))):
                        mask_2d[iy, ix] = True

    log.info("ROI subset rows %d:%d cols %d:%d shape %s, pixels in ROI: %d",
             r0, r1, c0, c1, lat_sub.shape, int(mask_2d.sum()))
    if int(mask_2d.sum()) == 0:
        raise ValueError("ROI polygon contains zero grid cells — check shapefile/CRS.")
    info = {"lat_path": lat_path, "lon_path": lon_path,
            "row_slice": (r0, r1), "col_slice": (c0, c1)}
    return slice(r0, r1), slice(c0, c1), mask_2d, lat_sub, lon_sub, info


def create_nc(nc_path, lat_2d, lon_2d, mask_2d, grid_info=None,
              row_slice=None, col_slice=None,
              avail_float_vars=None, avail_flag_vars=None):
    """Create a CF-1.8 compliant daily L3E output file."""
    ny, nx = lat_2d.shape
    float_vars = [v[0] for v in FLOAT_TARGETS] if avail_float_vars is None else list(avail_float_vars)
    flag_vars = [v[0] for v in FLAG_TARGETS] if avail_flag_vars is None else list(avail_flag_vars)

    ds = nc4.Dataset(nc_path, "w", format="NETCDF4")

    ds.createDimension("time", None)
    ds.createDimension("y", ny)
    ds.createDimension("x", nx)

    # --- time ---
    t_var = ds.createVariable("time", "f8", ("time",))
    t_var.units = TIME_UNITS
    t_var.calendar = TIME_CALENDAR
    t_var.axis = "T"
    t_var.standard_name = "time"
    t_var.long_name = "Daily composite center time (12:00 UTC) of SPL3SMP_E granule"

    # --- auxiliary 2D coordinates (native EASE-2 grid, no regridding) ---
    lat_var = ds.createVariable("lat", "f4", ("y", "x"), zlib=True, complevel=1,
                                fill_value=FILL_FLOAT)
    lat_var[:] = np.where(np.isfinite(lat_2d), lat_2d, FILL_FLOAT)
    lat_var.units = "degrees_north"
    lat_var.standard_name = "latitude"
    lat_var.long_name = "Latitude of EASE-Grid 2.0 9 km cell center"

    lon_var = ds.createVariable("lon", "f4", ("y", "x"), zlib=True, complevel=1,
                                fill_value=FILL_FLOAT)
    lon_var[:] = np.where(np.isfinite(lon_2d), lon_2d, FILL_FLOAT)
    lon_var.units = "degrees_east"
    lon_var.standard_name = "longitude"
    lon_var.long_name = "Longitude of EASE-Grid 2.0 9 km cell center"

    # --- ROI mask ---
    m_var = ds.createVariable("roi_mask", "i1", ("y", "x"), zlib=True, complevel=1)
    m_var[:] = mask_2d.astype(np.int8)
    m_var.long_name = "Region of interest mask (1=inside ROI, 0=outside)"
    m_var.flag_values = np.array([0, 1], dtype="i1")
    m_var.flag_meanings = "outside_roi inside_roi"
    m_var.coordinates = "lat lon"

    # --- data variables ---
    chunk_3d = (16, min(256, ny), min(256, nx))
    for vname in float_vars:
        meta = VAR_META.get(vname, {})
        v = ds.createVariable(vname, "f4", ("time", "y", "x"),
                              fill_value=FILL_FLOAT, zlib=True,
                              complevel=COMPRESS_LEVEL, chunksizes=chunk_3d,
                              shuffle=True)
        v.long_name = meta.get("long_name", vname)
        if "standard_name" in meta:
            v.standard_name = meta["standard_name"]
        v.units = meta.get("units", "1")
        v.valid_min = meta.get("valid_min", np.float32(0.0))
        v.valid_max = meta.get("valid_max", np.float32(1.0))
        v.missing_value = FILL_FLOAT
        v.grid_mapping = "crs"
        v.coordinates = "lat lon"
        if meta.get("ancillary"):
            v.ancillary_variables = meta["ancillary"]
        v.source = f"NASA SPL3SMP_E {'AM' if vname.endswith('_am') else 'PM'} DCA retrieval"
        v.cell_methods = "time: point area: point"

    for vname in flag_vars:
        fmeta = FLAG_META.get(vname, {})
        v = ds.createVariable(vname, "u2", ("time", "y", "x"),
                              fill_value=FILL_UINT16, zlib=True,
                              complevel=COMPRESS_LEVEL, chunksizes=chunk_3d,
                              shuffle=True)
        v.long_name = fmeta.get("long_name", vname)
        v.units = "1"
        v.valid_range = np.array([0, 65533], dtype="u2")
        v.missing_value = FILL_UINT16
        v.flag_values = np.array([0, 1], dtype="u2")
        v.flag_masks = fmeta.get("masks", "1 2")
        v.flag_meanings = fmeta.get("meanings", "flag_set flag_unset")
        v.coordinates = "lat lon"
        v.grid_mapping = "crs"

    # --- CRS / grid mapping (EPSG:6933) ---
    crs_var = ds.createVariable("crs", "i4", ())
    crs_var.grid_mapping_name = "lambert_cylindrical_equal_area"
    crs_var.longitude_of_central_meridian = 0.0
    crs_var.standard_parallel = 30.0
    crs_var.false_easting = 0.0
    crs_var.false_northing = 0.0
    crs_var.semi_major_axis = 6378137.0
    crs_var.inverse_flattening = 298.257223563
    crs_var.spatial_epsg = "6933"
    crs_var.spatial_ref = EASE2_PROJ4
    crs_var.crs_wkt = EASE2_WKT
    crs_var.long_name = "NSIDC EASE-Grid 2.0 Global 9 km (EPSG:6933)"

    # --- global attributes ---
    g_lat = lat_2d[mask_2d] if mask_2d.any() else lat_2d[np.isfinite(lat_2d)]
    g_lon = lon_2d[mask_2d] if mask_2d.any() else lon_2d[np.isfinite(lon_2d)]
    ds.Conventions = "CF-1.8"
    ds.title = "SMAP Enhanced L3 Radiometer Soil Moisture + VOD (SPL3SMP_E) — Ogallala ROI Subset"
    ds.summary = ("Daily subset of SPL3SMP_E (DCA soil moisture, VOD/vegetation optical depth, "
                  "retrieval quality and surface flags; AM descending + PM ascending) on the native "
                  "EASE-Grid 2.0 9 km grid, spatially subset to the Ogallala/High Plains aquifer ROI. "
                  "No interpolation or regridding applied; out-of-ROI cells are _FillValue.")
    ds.source = "NASA SMAP Enhanced L3 Radiometer Global Daily 9 km EASE-Grid Soil Moisture (SPL3SMP_E), DCA baseline"
    ds.institution = "NASA NSIDC DAAC"
    ds.references = ("O'Neill et al. SPL3SMP_E User Guide v6 (https://doi.org/10.5067/M20OXIZHY3RJ); "
                     "https://nsidc.org/data/spl3smp_e")
    ds.history = f"Created {datetime.now(timezone.utc).isoformat()} by smap_l3smp_e_extractor.py"
    ds.product_version = "SPL3SMP_E v005/v006 compatible (auto-detected per granule)"
    ds.cdm_data_type = "Grid"
    ds.featureType = "grid"
    ds.standard_name_vocabulary = "CF Standard Name Table v79"
    ds.geospatial_lat_min = float(np.nanmin(g_lat))
    ds.geospatial_lat_max = float(np.nanmax(g_lat))
    ds.geospatial_lon_min = float(np.nanmin(g_lon))
    ds.geospatial_lon_max = float(np.nanmax(g_lon))
    ds.geospatial_lat_units = "degrees_north"
    ds.geospatial_lon_units = "degrees_east"
    ds.geospatial_vertical_positive = "up"
    ds.time_coverage_start = "2015-03-31T12:00:00Z"
    ds.time_coverage_resolution = "P1D"
    ds.naming_authority = "NASA NSIDC DAAC"
    ds.creator_name = "smap_l3smp_e_extractor.py"
    if grid_info:
        ds.comment = (f"Native grid {grid_info.get('grid_shape', '')}; "
                      f"geolocation {grid_info.get('lat_path')}/{grid_info.get('lon_path')}; "
                      f"row slice {row_slice}, col slice {col_slice}. "
                      "AM=descending ~06:00 local, PM=ascending ~18:00 local. "
                      "Use retrieval_qual_flag bit 0 == 0 for recommended quality.")
    ds.close()


def _resolve_in_group(f, group, candidates):
    """Return (dataset_path, dataset) for first candidate present in group."""
    if group not in f:
        return None, None
    g = f[group]
    for name in candidates:
        if name in g and isinstance(g[name], h5py.Dataset):
            return f"{group}/{name}", g[name]
    return None, None


def validate_h5(filepath):
    """Readable + has geolocation + at least one SM/VOD/flag dataset."""
    try:
        lat_p, lon_p, shape, _ = _find_latlon(filepath)
        with h5py.File(filepath, "r") as f:
            _ = f[lat_p][:1, :1]
            _ = f[lon_p][:1, :1]
            hits = 0
            for _out, grp, cands in FLOAT_TARGETS + FLAG_TARGETS:
                p, d = _resolve_in_group(f, grp, cands)
                if d is not None:
                    _ = d[:1, :1]  # force read to catch silent corruption
                    hits += 1
            # Require at minimum one SM + one VOD + one flag resolvable
            return hits >= 3
    except Exception:
        return False


def _read_subset(dset, row_slice, col_slice):
    arr = dset[row_slice, col_slice]
    # Dereference object-reference pointer datasets if encountered (rare,
    # since we prefer *_dca). If references, give up on this dataset.
    if arr.dtype.kind == "O":
        raise TypeError(f"Dataset {dset.name} holds object references; use *_dca instead.")
    return arr


def extract_granule(h5_path, row_slice, col_slice, mask_2d):
    """Read one daily granule -> dict {output_var: 2D array} + availability.

    Floats: cast to float32, out-of-ROI -> FILL_FLOAT.
    Flags:  read as uint16, out-of-ROI -> FILL_UINT16.
    Missing optional vars (e.g. vod_error) are skipped, not fatal.
    """
    data, avail_f, avail_q = {}, [], []
    with h5py.File(h5_path, "r") as f:
        for out_name, group, cands in FLOAT_TARGETS:
            p, d = _resolve_in_group(f, group, cands)
            if d is None:
                continue  # optional (e.g. error fields absent in some versions)
            try:
                arr = _read_subset(d, row_slice, col_slice).astype(np.float32)
            except TypeError as e:
                log.warning("%s: %s — skipping", out_name, e)
                continue
            # Normalize source fill (-9999) + non-finite to our fill, mask ROI
            arr[~np.isfinite(arr)] = FILL_FLOAT
            arr[~mask_2d] = FILL_FLOAT
            data[out_name] = arr
            avail_f.append(out_name)
        for out_name, group, cands in FLAG_TARGETS:
            p, d = _resolve_in_group(f, group, cands)
            if d is None:
                continue
            arr = np.asarray(_read_subset(d, row_slice, col_slice)).astype(np.uint16)
            # Map any fill-like values (>=65534) uniformly to FILL_UINT16
            arr[arr >= np.uint16(65534)] = FILL_UINT16
            arr[~mask_2d] = FILL_UINT16
            data[out_name] = arr
            avail_q.append(out_name)
    return data, avail_f, avail_q


def ensure_nc_vars(ds, avail_float, avail_flag):
    """Add any newly-seen optional vars to an existing file (version drift)."""
    ny = len(ds.dimensions["y"])
    nx = len(ds.dimensions["x"])
    chunk_3d = (16, min(256, ny), min(256, nx))
    added = []
    for vname in avail_float:
        if vname in ds.variables:
            continue
        meta = VAR_META.get(vname, {})
        v = ds.createVariable(vname, "f4", ("time", "y", "x"), fill_value=FILL_FLOAT,
                              zlib=True, complevel=COMPRESS_LEVEL,
                              chunksizes=chunk_3d, shuffle=True)
        v.long_name = meta.get("long_name", vname)
        if "standard_name" in meta:
            v.standard_name = meta["standard_name"]
        v.units = meta.get("units", "1")
        v.valid_min = meta.get("valid_min", np.float32(0.0))
        v.valid_max = meta.get("valid_max", np.float32(1.0))
        v.missing_value = FILL_FLOAT
        v.grid_mapping = "crs"
        v.coordinates = "lat lon"
        if meta.get("ancillary"):
            v.ancillary_variables = meta["ancillary"]
        added.append(vname)
    for vname in avail_flag:
        if vname in ds.variables:
            continue
        fmeta = FLAG_META.get(vname, {})
        v = ds.createVariable(vname, "u2", ("time", "y", "x"), fill_value=FILL_UINT16,
                              zlib=True, complevel=COMPRESS_LEVEL,
                              chunksizes=chunk_3d, shuffle=True)
        v.long_name = fmeta.get("long_name", vname)
        v.units = "1"
        v.valid_range = np.array([0, 65533], dtype="u2")
        v.missing_value = FILL_UINT16
        v.flag_masks = fmeta.get("masks", "1 2")
        v.flag_meanings = fmeta.get("meanings", "flag_set flag_unset")
        v.coordinates = "lat lon"
        v.grid_mapping = "crs"
        added.append(vname)
    if added:
        log.info("Added %d variables to existing file for version compatibility: %s", len(added), added)


def get_md5(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        while chunk := f.read(10485760):
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst):
    src_md5 = get_md5(src)
    for attempt in range(1, 4):
        try:
            with open(src, "rb") as fs, open(dst, "wb") as fd:
                shutil.copyfileobj(fs, fd, length=10485760)
                fd.flush()
                os.fsync(fd.fileno())
            if os.path.getsize(src) == os.path.getsize(dst) and src_md5 == get_md5(dst):
                log.info("Drive copy verified (%.1f MiB)", os.path.getsize(dst) / 1048576)
                return True
            raise IOError("Size or MD5 mismatch")
        except Exception as e:
            log.error("Drive copy attempt %d failed: %s", attempt, e)
            if os.path.exists(dst):
                os.remove(dst)
            time.sleep(5 * attempt)
    raise RuntimeError(f"Drive copy failed after 3 attempts")


# ============================================================================
# PIPELINE
# ============================================================================
def run_pipeline():
    for _d in (LOCAL_WORK, LOCAL_NC, DRIVE_OUT_DIR):
        os.makedirs(_d, exist_ok=True)
    earthaccess.login(strategy="interactive")
    roi = load_roi(SHAP_PATH)

    log.info("Fetching sample SPL3SMP_E granule for grid detection...")
    res = earthaccess.search_data(short_name=SHORT_NAME,
                                  temporal=("2020-01-01", "2020-01-02"), count=5)
    if not res:
        raise RuntimeError("No SPL3SMP_E granules found for sample grid detection.")
    sample = None
    for r in res:
        try:
            dl = safe_download([r], LOCAL_WORK, threads=1)
            if not dl:
                continue
            cand = str(dl[0])
            _find_latlon(cand)  # must be readable L3E
            sample = cand
            break
        except Exception as e:
            log.warning("Sample candidate unusable: %s", e)
            continue
    if sample is None:
        raise RuntimeError("Could not download a usable SPL3SMP_E sample granule.")

    row_slice, col_slice, mask_2d, lat_sub, lon_sub, grid_info = build_mask(sample, roi)
    with h5py.File(sample, "r") as f:
        for grp in (AM_GROUP, PM_GROUP):
            if grp in f:
                log.info("Sample %s datasets: %s", grp, sorted(list(f[grp].keys()))[:40])
    grid_info["grid_shape"] = f"{lat_sub.shape} subset of EASE-2 9km global 1624x3856"
    os.remove(sample)

    # --- resume: local file survives only if runtime did not reset;
    # Drive copy is the source of truth after a restart. This block restores
    # it (any capitalization: FULL / Full) and reads the last date from it,
    # so no completed days are re-downloaded. Nothing is lost as long as the
    # Drive .nc exists — progress = number of timesteps already flushed. ---
    local_nc = os.path.join(LOCAL_NC, OUTPUT_FILENAME)
    if SINGLE_FILE_MODE and not os.path.exists(local_nc):
        latest_drive = get_latest_drive_backup(DRIVE_OUT_DIR)
        if latest_drive:
            log.info("Restoring latest backup from Drive: %s (%.1f MiB)...",
                     os.path.basename(latest_drive),
                     os.path.getsize(latest_drive) / 1048576)
            shutil.copy2(latest_drive, local_nc)
            # If Drive name capitalization differs (Full vs FULL), normalize
            # the local copy to OUTPUT_FILENAME (already the case here).
        else:
            log.info("No Drive backup found — starting from mission start.")

    start_dt = MISSION_START
    cur_nc = local_nc
    if local_nc and os.path.exists(local_nc):
        last_dt, n_times = read_last_timestep(local_nc)
        if last_dt is not None:
            start_dt = last_dt + timedelta(days=1)
            log.info("Resuming %s at %s (existing timesteps: %d). "
                     "Progress is SAFE — last date in file: %s.",
                     cur_nc, start_dt.date(), n_times, last_dt.date())
        else:
            try:
                log.warning("Existing file empty/corrupt (0 timesteps). Recreating.")
                os.remove(local_nc)
            except OSError:
                pass
            create_nc(cur_nc, lat_sub, lon_sub, mask_2d, grid_info,
                      row_slice, col_slice)
    else:
        create_nc(cur_nc, lat_sub, lon_sub, mask_2d, grid_info,
                  row_slice, col_slice)

    script_start = time.time()
    runtime_ckpt = 0
    next_ckpt = script_start + CHECKPOINT_SECONDS

    while start_dt.replace(tzinfo=None) < datetime.now():
        end_dt = min(start_dt + timedelta(days=BATCH_DATES), datetime.now())
        log.info("Batch: %s to %s", start_dt.date(), end_dt.date())

        # NOTE: earthaccess.download([]) raises
        #   ValueError: List of URLs or DataGranule instances expected
        # so NEVER call download with an empty list. An empty search result
        # happens on real data gaps (e.g. Safe Mode 2019-06-19..2019-07-23,
        # 2022-08-06..2022-09-20), on CMR pagination hiccups, or when the
        # window reaches "now" with nothing posted yet. It is NOT fatal:
        # just advance the window and continue.
        results = safe_search(start_dt, end_dt, retries=3)
        if not results:
            log.warning("Search returned 0 granules for %s..%s — "
                        "skipping window (data gap or CMR hiccup), no progress lost.",
                        start_dt.date(), end_dt.date())
            start_dt = end_dt + timedelta(days=1)
            gc.collect()
            continue
        downloaded = safe_download(results, LOCAL_WORK, threads=N_THREADS)
        if not downloaded:
            log.warning("Download returned 0 files for %s..%s — skipping window.",
                        start_dt.date(), end_dt.date())
            start_dt = end_dt + timedelta(days=1)
            gc.collect()
            continue

        current_batch = []
        for f in downloaded:
            dt = parse_dt(str(f))
            if dt:
                current_batch.append((dt, str(f)))
            else:
                log.warning("Skipping file with unparseable name: %s", os.path.basename(str(f)))

        # --- validation + targeted redownload ---
        valid_granules, retry = [], 0
        pending = list(current_batch)
        while pending and retry < MAX_RETRIES:
            good, bad = [], []
            log.info("Validation pass %d: checking %d files...", retry + 1, len(pending))
            for dt, p in pending:
                (good if validate_h5(p) else bad).append((dt, p))
                if (dt, p) in bad:
                    try:
                        os.remove(p)
                    except OSError:
                        pass
            valid_granules.extend(good)
            pending = []
            if not bad:
                log.info("All files validated.")
                break
            retry += 1
            log.warning("Redownloading %d corrupted files (attempt %d/%d)...",
                        len(bad), retry, MAX_RETRIES)
            for dt, _ in bad:
                try:
                    r = safe_search(dt - timedelta(hours=12), dt + timedelta(hours=12), retries=2)
                    for f in safe_download(r, LOCAL_WORK, threads=1):
                        if parse_dt(str(f)) == dt:
                            pending.append((dt, str(f)))
                            break
                except Exception as e:
                    log.error("Redownload failed for %s: %s", dt.date(), e)
        if pending:
            log.error("%d files still corrupt after %d retries — skipped.", len(pending), MAX_RETRIES)

        valid_granules.sort(key=lambda x: x[0])
        log.info("Extracting %d valid daily granules.", len(valid_granules))
        if not valid_granules:
            log.warning("No valid granules in batch %s..%s — advancing.",
                        start_dt.date(), end_dt.date())
            for f in glob.glob(os.path.join(LOCAL_WORK, "*.h5")):
                try:
                    os.remove(f)
                except OSError:
                    pass
            start_dt = end_dt + timedelta(days=1)
            gc.collect()
            continue

        ds = nc4.Dataset(cur_nc, "a")
        dates_flushed, last_day, processed = 0, None, 0

        for dt, h5_path in valid_granules:
            if dt < start_dt:
                continue
            try:
                data, avail_f, avail_q = extract_granule(h5_path, row_slice, col_slice, mask_2d)
                if not data:
                    log.warning("No extractable variables in %s — skipping.", os.path.basename(h5_path))
                    continue
                ensure_nc_vars(ds, avail_f, avail_q)
                t_idx = len(ds.variables["time"])
                t_num = nc4.date2num(dt, ds.variables["time"].units, ds.variables["time"].calendar)
                ds.variables["time"][t_idx] = t_num
                for vname, arr in data.items():
                    if vname not in ds.variables:
                        continue
                    if arr.dtype == np.uint16:
                        ds.variables[vname][t_idx, :, :] = arr
                    else:
                        ds.variables[vname][t_idx, :, :] = arr.astype(np.float32)
                processed += 1
                if processed % 5 == 0:
                    ds.sync()

                if time.time() >= next_ckpt:
                    runtime_ckpt += 1
                    ckpt_hours = runtime_ckpt * CHECKPOINT_SECONDS // 3600
                    ckpt_name = f"SMAP_L3E_ogallala_{ckpt_hours}hr.nc"
                    log.info("1-h runtime checkpoint: saving %s ...", ckpt_name)
                    ds.sync()
                    ds.close()
                    try:
                        robust_drive_copy(cur_nc, os.path.join(DRIVE_OUT_DIR, ckpt_name))
                    except Exception as e:
                        log.error("Checkpoint copy failed: %s", e)
                    ds = nc4.Dataset(cur_nc, "a")
                    next_ckpt = time.time() + CHECKPOINT_SECONDS
                    gc.collect()

                day = dt.strftime("%Y%m%d")
                if day != last_day:
                    dates_flushed += 1
                    last_day = day
                if dates_flushed >= FLUSH_DATES:
                    ds.sync()
                    # refresh time coverage end for CF discoverability
                    try:
                        ds.time_coverage_end = dt.strftime("%Y-%m-%dT12:00:00Z")
                        ds.history = (getattr(ds, "history", "")
                                      + f"; appended through {dt.date()} {datetime.now(timezone.utc).isoformat()}")
                    except Exception:
                        pass
                    log.info("Flushed %d timesteps to disk.", processed)
                    if SINGLE_FILE_MODE and dates_flushed >= BACKUP_DATES:
                        log.info("Backing up monolithic file to Drive...")
                        ds.close()
                        robust_drive_copy(cur_nc, os.path.join(DRIVE_OUT_DIR, os.path.basename(cur_nc)))
                        ds = nc4.Dataset(cur_nc, "a")
                        dates_flushed = 0
                    else:
                        dates_flushed = 0
                    gc.collect()

                os.remove(h5_path)
                del data
            except Exception as e:
                log.error("Error processing %s: %s — skipping.", os.path.basename(h5_path), e)
                if os.path.exists(h5_path):
                    try:
                        os.remove(h5_path)
                    except OSError:
                        pass
                continue

        try:
            ds.time_coverage_end = end_dt.strftime("%Y-%m-%dT12:00:00Z")
        except Exception:
            pass
        ds.close()

        log.info("Batch complete (%d days). Backing up to Drive...", processed)
        robust_drive_copy(cur_nc, os.path.join(DRIVE_OUT_DIR, os.path.basename(cur_nc)))

        for f in glob.glob(os.path.join(LOCAL_WORK, "*.h5")):
            try:
                os.remove(f)
            except OSError:
                pass
        start_dt = end_dt + timedelta(days=1)
        gc.collect()


if __name__ == "__main__":
    run_pipeline()

Enter your Earthdata Login username: watcher69
Enter your Earthdata password: ··········


  0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_1666/1062615376.py:388: DeprecationWarning: The 'shapely.vectorized.contains' function is deprecated and will be removed a future version. Use 'shapely.contains_xy' instead (available since shapely 2.0.0).
  mask_2d = vec_contains(roi, lon_sub, lat_sub)


QUEUEING TASKS | :   0%|          | 0/9 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/9 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
!curl -sI --max-time 10 https://www.google.com | head -1        # Is the VM online at all?
!curl -sI --max-time 10 https://urs.earthdata.nasa.gov | head -1   # Can it reach Earthdata Login?
!curl -sI --max-time 10 https://cmr.earthdata.nasa.gov | head -1   # Can it reach CMR?

import socket
print(socket.getaddrinfo('urs.earthdata.nasa.gov', 443))  # check for IPv6 (AF_INET6) entries

HTTP/2 200 
HTTP/1.1 200 OK
HTTP/2 301 
[(<AddressFamily.AF_INET: 2>, <SocketKind.SOCK_STREAM: 1>, 6, '', ('198.118.243.33', 443)), (<AddressFamily.AF_INET: 2>, <SocketKind.SOCK_DGRAM: 2>, 17, '', ('198.118.243.33', 443)), (<AddressFamily.AF_INET: 2>, <SocketKind.SOCK_RAW: 3>, 0, '', ('198.118.243.33', 443)), (<AddressFamily.AF_INET6: 10>, <SocketKind.SOCK_STREAM: 1>, 6, '', ('2001:4d0:241a:4081::89', 443, 0, 0)), (<AddressFamily.AF_INET6: 10>, <SocketKind.SOCK_DGRAM: 2>, 17, '', ('2001:4d0:241a:4081::89', 443, 0, 0)), (<AddressFamily.AF_INET6: 10>, <SocketKind.SOCK_RAW: 3>, 0, '', ('2001:4d0:241a:4081::89', 443, 0, 0))]
